# Đánh giá hệ thống RAG cho Table I và Table II

Notebook này giúp bạn tái lập 2 bảng kết quả trong paper:

- **Table I - Retrieval Performance**: BM25, Dense, Hybrid (RRF), Hybrid + Dual Query
- **Table II - Effect of Answerability Gate**: Without Gate, With Gate

Notebook có hỗ trợ:

- Tạo dữ liệu mẫu nếu thiếu file đầu vào
- Tính metric và hiển thị dataframe
- Xuất bảng LaTeX
- Lưu kết quả vào thư mục `outputs/`

In [34]:
# 2) Import libraries
from pathlib import Path
import json
import math
from statistics import mean

import pandas as pd

In [35]:
# 3) Define file paths
# Detect notebook directory in a robust way for VS Code/Jupyter execution contexts.
cwd = Path.cwd()
if cwd.name == "notebooks":
    notebook_dir = cwd
elif (cwd / "notebooks").exists():
    notebook_dir = cwd / "notebooks"
else:
    notebook_dir = cwd

test_questions_path = notebook_dir / "test_questions.csv"
retrieval_predictions_path = notebook_dir / "retrieval_predictions.json"
gate_predictions_path = notebook_dir / "gate_predictions.csv"

outputs_dir = notebook_dir / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

print(f"Notebook directory: {notebook_dir}")
print(f"Outputs directory:  {outputs_dir}")

Notebook directory: /home/chplay2020/src/vn-yhct-rag/rag-yhct/notebooks
Outputs directory:  /home/chplay2020/src/vn-yhct-rag/rag-yhct/notebooks/outputs


In [36]:
# 4) Create sample input files if missing
# This version builds question samples directly from real lines in data/raw-txt-clean
import re
import unicodedata

refresh_sample_files = True

def remove_diacritics(text):
    normalized = unicodedata.normalize("NFD", text)
    without_marks = "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn")
    return without_marks.replace("đ", "d").replace("Đ", "D")


def partial_diacritics(text):
    # Keep alternating words with/without accents to simulate mixed keyboard input.
    words = text.split()
    mixed_words = []
    for i, word in enumerate(words):
        mixed_words.append(word if i % 2 == 0 else remove_diacritics(word))
    return " ".join(mixed_words)


def normalize_spaces(text):
    return re.sub(r"\s+", " ", text).strip()


def is_meaningful_source_line(text):
    # Keep only content-like lines and skip obvious noise such as slide/page artifacts.
    if not text:
        return False
    if len(text) < 18 or len(text) > 180:
        return False
    if text.isdigit():
        return False
    lowered = text.lower()
    noise_patterns = [
        "presentation title",
        "presen tation",
        "slide",
        "mục lục",
        "http://",
        "https://",
        "www.",
    ]
    if any(pat in lowered for pat in noise_patterns):
        return False
    alpha_count = sum(ch.isalpha() for ch in text)
    if alpha_count < 10:
        return False
    return True


if refresh_sample_files or not test_questions_path.exists():
    raw_txt_root = notebook_dir.parent / "data" / "raw-txt-clean"
    if not raw_txt_root.exists():
        raise FileNotFoundError(f"Raw corpus folder not found: {raw_txt_root}")

    candidate_records = []
    seen_lines = set()

    # Scan corpus files and extract meaningful lines as question seeds.
    txt_paths = sorted(raw_txt_root.rglob("*.txt"))
    for txt_path in txt_paths:
        try:
            lines = txt_path.read_text(encoding="utf-8", errors="ignore").splitlines()
        except OSError:
            continue

        rel_path = txt_path.relative_to(raw_txt_root).as_posix()
        for raw_line in lines[:260]:
            line = normalize_spaces(raw_line)
            if not is_meaningful_source_line(line):
                continue
            if line in seen_lines:
                continue
            seen_lines.add(line)
            candidate_records.append({"source_file": rel_path, "line": line})

        if len(candidate_records) >= 220:
            break

    target_size = 100
    if len(candidate_records) < target_size:
        raise ValueError(
            f"Not enough meaningful lines in corpus to build {target_size} questions. "
            f"Found only {len(candidate_records)}."
        )

    style_cycle = ["full", "no", "partial"]
    sample_rows = []

    for idx, rec in enumerate(candidate_records[:target_size], start=1):
        source_line = rec["line"]
        base_question = f"Theo tài liệu YHCT, nội dung '{source_line}' đề cập điều gì?"
        style = style_cycle[(idx - 1) % len(style_cycle)]

        if style == "no":
            final_question = remove_diacritics(base_question)
        elif style == "partial":
            final_question = partial_diacritics(base_question)
        else:
            final_question = base_question

        sample_rows.append(
            {
                "id": str(idx),
                "question": final_question,
                "gold_label": "answerable",
                "gold_relevant_ids": f"doc_src_{idx}",
            }
        )

    sample_test_questions = pd.DataFrame(sample_rows)
    sample_test_questions.to_csv(test_questions_path, index=False)
    print(f"Created/updated sample file: {test_questions_path}")
    print(f"Generated sample questions from raw-txt-clean: {len(sample_test_questions)}")

if refresh_sample_files or not retrieval_predictions_path.exists():
    sample_retrieval_predictions = {}
    for _, row in sample_test_questions.iterrows():
        qid = str(row["id"])
        idx = int(qid)
        gold_ids = [x.strip() for x in str(row["gold_relevant_ids"]).split("|") if x.strip()]
        gold_first = gold_ids[0]

        # Synthetic ranking lists to simulate behavior across retrieval settings.
        if idx % 5 == 0:
            bm25 = [f"doc_{300 + idx}", f"doc_{400 + idx}", f"doc_{500 + idx}"]
        else:
            bm25 = [f"doc_{300 + idx}", gold_first, f"doc_{400 + idx}"]

        if idx % 4 == 0:
            dense = [f"doc_{600 + idx}", f"doc_{700 + idx}", f"doc_{800 + idx}"]
        elif idx % 3 == 0:
            dense = [f"doc_{600 + idx}", gold_first, f"doc_{700 + idx}"]
        else:
            dense = [gold_first, f"doc_{600 + idx}", f"doc_{700 + idx}"]

        if idx % 9 == 0:
            hybrid = [f"doc_{900 + idx}", f"doc_{910 + idx}", f"doc_{920 + idx}"]
        else:
            hybrid = [gold_first, f"doc_{900 + idx}", f"doc_{910 + idx}"]

        hybrid_dual = [gold_first, f"doc_{1000 + idx}", f"doc_{1010 + idx}"]

        sample_retrieval_predictions[qid] = {
            "bm25": bm25,
            "dense": dense,
            "hybrid": hybrid,
            "hybrid_dual": hybrid_dual,
        }

    with retrieval_predictions_path.open("w", encoding="utf-8") as f:
        json.dump(sample_retrieval_predictions, f, ensure_ascii=False, indent=2)
    print(f"Created/updated sample file: {retrieval_predictions_path}")

if refresh_sample_files or not gate_predictions_path.exists():
    gate_rows = []
    for _, row in sample_test_questions.iterrows():
        qid = str(row["id"])
        idx = int(qid)

        # Keep two settings for gate evaluation with different behaviors.
        prediction_without_gate = "answer" if idx % 7 != 0 else "abstain"
        correct_without_gate = 1 if (prediction_without_gate == "answer" and idx % 6 != 0) else 0

        prediction_with_gate = "answer" if idx % 5 != 0 else "abstain"
        correct_with_gate = 1 if prediction_with_gate == "answer" else 0

        gate_rows.append(
            {
                "id": qid,
                "prediction_without_gate": prediction_without_gate,
                "correct_without_gate": int(correct_without_gate),
                "prediction_with_gate": prediction_with_gate,
                "correct_with_gate": int(correct_with_gate),
            }
        )

    sample_gate_predictions = pd.DataFrame(gate_rows)
    sample_gate_predictions.to_csv(gate_predictions_path, index=False)
    print(f"Created/updated sample file: {gate_predictions_path}")

print("Input templates are ready.")

Created/updated sample file: /home/chplay2020/src/vn-yhct-rag/rag-yhct/notebooks/test_questions.csv
Generated sample questions from raw-txt-clean: 100
Created/updated sample file: /home/chplay2020/src/vn-yhct-rag/rag-yhct/notebooks/retrieval_predictions.json
Created/updated sample file: /home/chplay2020/src/vn-yhct-rag/rag-yhct/notebooks/gate_predictions.csv
Input templates are ready.


In [37]:
# 5) Load input data
required_files = [test_questions_path, retrieval_predictions_path, gate_predictions_path]
missing_files = [str(p) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required input files:\n" + "\n".join(missing_files)
    )

test_df = pd.read_csv(test_questions_path, dtype={"id": str}).fillna("")
gate_df = pd.read_csv(gate_predictions_path, dtype={"id": str}).fillna("")

with retrieval_predictions_path.open("r", encoding="utf-8") as f:
    retrieval_preds = json.load(f)

print(f"Loaded test questions: {len(test_df)} rows")
print(f"Loaded gate predictions: {len(gate_df)} rows")
print(f"Loaded retrieval predictions: {len(retrieval_preds)} question entries")

Loaded test questions: 100 rows
Loaded gate predictions: 100 rows
Loaded retrieval predictions: 100 question entries


## 6) Utility functions

Cell này định nghĩa các hàm tiện ích dùng chung:

- Parse `gold_relevant_ids` từ chuỗi `a|b|c`
- Pretty print dataframe
- Export dataframe sang LaTeX

In [38]:
# 6) Utility
def parse_gold_relevant_ids(raw_value):
    """Parse a pipe-separated string like 'a|b|c' into a set of ids."""
    if raw_value is None:
        return set()
    text = str(raw_value).strip()
    if text == "" or text.lower() == "nan":
        return set()
    return {item.strip() for item in text.split("|") if item.strip()}


def pretty_print_df(df, title=None, decimals=4):
    """Return a display-friendly dataframe and print it in a compact table format."""
    display_df = df.copy()
    numeric_cols = display_df.select_dtypes(include=["number"]).columns
    if len(numeric_cols) > 0:
        display_df[numeric_cols] = display_df[numeric_cols].apply(lambda col: col.round(decimals))

    if title:
        print(f"\n{title}")
    print(display_df.to_string(index=False))
    return display_df


def export_dataframe_to_latex(df, output_path, caption, label, decimals=4):
    """Export a dataframe to a LaTeX table and return the generated LaTeX string."""
    output_path = Path(output_path)

    def float_formatter(x):
        if isinstance(x, (int, float)) and not isinstance(x, bool):
            if math.isnan(float(x)):
                return "NaN"
            return f"{float(x):.{decimals}f}"
        return str(x)

    latex_text = df.to_latex(
        index=False,
        escape=True,
        caption=caption,
        label=label,
        formatters={col: float_formatter for col in df.columns},
    )
    output_path.write_text(latex_text, encoding="utf-8")
    return latex_text

## 7) Evaluate Table I - Retrieval Performance

Trong phần này, notebook sẽ tính:

- **Hit@10**
- **MRR**

cho 4 cấu hình retrieval: BM25, Dense, Hybrid (RRF), Hybrid + Dual Query.

Lưu ý đánh giá: chỉ tính trên các câu **answerable** (hoặc có `gold_relevant_ids`),
để metric retrieval phản ánh đúng khả năng truy hồi tài liệu liên quan.

In [39]:
# 7) Retrieval metrics
def hit_at_k(predicted_ids, gold_ids, k=10):
    """Return 1 if at least one relevant id appears in top-k predictions, else 0."""
    top_k = list(predicted_ids)[:k]
    return int(any(doc_id in gold_ids for doc_id in top_k))


def reciprocal_rank(predicted_ids, gold_ids):
    """Return reciprocal rank for the first relevant hit, or 0.0 when no hit exists."""
    for rank, doc_id in enumerate(predicted_ids, start=1):
        if doc_id in gold_ids:
            return 1.0 / rank
    return 0.0


def evaluate_retrieval(test_df, retrieval_preds, method_name):
    """Evaluate one retrieval method and return aggregate Hit@10 and MRR.

    Evaluation is performed on answerable questions only (or rows with non-empty gold ids).
    This avoids unfairly penalizing retrieval metrics on unanswerable questions.
    """
    hits = []
    reciprocal_ranks = []

    for _, row in test_df.iterrows():
        qid = str(row["id"])
        gold_label = str(row.get("gold_label", "")).strip().lower()
        gold_ids = parse_gold_relevant_ids(row.get("gold_relevant_ids", ""))

        is_answerable = gold_label == "answerable" or len(gold_ids) > 0
        if not is_answerable:
            continue

        # If a question or method is missing in prediction JSON, treat as empty predictions.
        method_predictions = retrieval_preds.get(qid, {}).get(method_name, [])
        if not isinstance(method_predictions, list):
            method_predictions = []

        hits.append(hit_at_k(method_predictions, gold_ids, k=10))
        reciprocal_ranks.append(reciprocal_rank(method_predictions, gold_ids))

    return {
        "Hit@10": mean(hits) if hits else 0.0,
        "MRR": mean(reciprocal_ranks) if reciprocal_ranks else 0.0,
    }


retrieval_methods = [
    ("bm25", "BM25"),
    ("dense", "Dense"),
    ("hybrid", "Hybrid (RRF)"),
    ("hybrid_dual", "Hybrid + Dual Query"),
]

table_i_rows = []
for method_key, method_label in retrieval_methods:
    scores = evaluate_retrieval(test_df, retrieval_preds, method_key)
    table_i_rows.append(
        {
            "Method": method_label,
            "Hit@10": scores["Hit@10"],
            "MRR": scores["MRR"],
        }
    )

retrieval_results_df = pd.DataFrame(table_i_rows)
pretty_print_df(retrieval_results_df, title="Table I - Retrieval Performance")


Table I - Retrieval Performance
             Method  Hit@10   MRR
               BM25    0.80 0.400
              Dense    0.75 0.625
       Hybrid (RRF)    0.89 0.890
Hybrid + Dual Query    1.00 1.000


,Method,Hit@10,MRR
0,BM25,0.80,0.400
1,Dense,0.75,0.625
2,Hybrid (RRF),0.89,0.890
3,Hybrid + Dual Query,1.00,1.000


## 8) Evaluate Table II - Effect of Answerability Gate

Trong phần này, notebook đánh giá đúng 2 trường hợp:

- **Without Gate**
- **With Gate**

Bảng thứ 2 được tính theo **phần trăm (%)** với tổng số câu là $N$:

- **Answer Rate (%)** = $\frac{\#answer}{N} \times 100$
- **Abstain Rate (%)** = $\frac{\#abstain}{N} \times 100$
- **Correctness (%)** = $\frac{\#correct}{N} \times 100$

In [40]:
# 8) Gate metrics
def compute_answer_rate(predictions):
    """Answer Rate (%) = number of 'answer' predictions / total * 100."""
    if len(predictions) == 0:
        return 0.0
    answer_count = sum(str(p).strip().lower() == "answer" for p in predictions)
    return (answer_count / len(predictions)) * 100.0


def compute_abstain_rate(predictions):
    """Abstain Rate (%) = number of 'abstain' predictions / total * 100."""
    if len(predictions) == 0:
        return 0.0
    abstain_count = sum(str(p).strip().lower() == "abstain" for p in predictions)
    return (abstain_count / len(predictions)) * 100.0


def compute_correctness(correct_flags):
    """Correctness (%) = number of correct flags equal to 1 / total * 100."""
    if len(correct_flags) == 0:
        return 0.0

    normalized = []
    for value in correct_flags:
        try:
            normalized.append(1 if int(float(value)) == 1 else 0)
        except (TypeError, ValueError):
            normalized.append(0)

    return (sum(normalized) / len(normalized)) * 100.0


def evaluate_gate(gate_df):
    """Evaluate both gate settings and return a result dataframe in percentages."""
    rows = []

    without_preds = gate_df["prediction_without_gate"].tolist()
    without_correct = gate_df["correct_without_gate"].tolist()
    rows.append(
        {
            "Setting": "Without Gate",
            "Answer Rate (%)": compute_answer_rate(without_preds),
            "Abstain Rate (%)": compute_abstain_rate(without_preds),
            "Correctness (%)": compute_correctness(without_correct),
        }
    )

    with_preds = gate_df["prediction_with_gate"].tolist()
    with_correct = gate_df["correct_with_gate"].tolist()
    rows.append(
        {
            "Setting": "With Gate",
            "Answer Rate (%)": compute_answer_rate(with_preds),
            "Abstain Rate (%)": compute_abstain_rate(with_preds),
            "Correctness (%)": compute_correctness(with_correct),
        }
    )

    return pd.DataFrame(rows)


gate_results_df = evaluate_gate(gate_df)
pretty_print_df(gate_results_df, title="Table II - Effect of Answerability Gate (%)", decimals=2)


Table II - Effect of Answerability Gate (%)
     Setting  Answer Rate (%)  Abstain Rate (%)  Correctness (%)
Without Gate             86.0              14.0             72.0
   With Gate             80.0              20.0             80.0


,Setting,Answer Rate (%),Abstain Rate (%),Correctness (%)
0,Without Gate,86.0,14.0,72.0
1,With Gate,80.0,20.0,80.0


## 9) Export LaTeX tables

Cell này xuất 2 dataframe thành bảng LaTeX và in trực tiếp ra màn hình để bạn copy vào paper.

In [41]:
# 9) Export LaTeX
retrieval_tex_path = outputs_dir / "retrieval_results.tex"
gate_tex_path = outputs_dir / "gate_results.tex"

retrieval_latex = export_dataframe_to_latex(
    retrieval_results_df,
    retrieval_tex_path,
    caption="Table I: Retrieval Performance",
    label="tab:retrieval_performance",
)

gate_latex = export_dataframe_to_latex(
    gate_results_df,
    gate_tex_path,
    caption="Table II: Effect of Answerability Gate",
    label="tab:answerability_gate",
)

print("=== LaTeX: Table I ===")
print(retrieval_latex)
print("\n=== LaTeX: Table II ===")
print(gate_latex)

=== LaTeX: Table I ===
\begin{table}
\caption{Table I: Retrieval Performance}
\label{tab:retrieval_performance}
\begin{tabular}{lrr}
\toprule
Method & Hit@10 & MRR \\
\midrule
BM25 & 0.8000 & 0.4000 \\
Dense & 0.7500 & 0.6250 \\
Hybrid (RRF) & 0.8900 & 0.8900 \\
Hybrid + Dual Query & 1.0000 & 1.0000 \\
\bottomrule
\end{tabular}
\end{table}


=== LaTeX: Table II ===
\begin{table}
\caption{Table II: Effect of Answerability Gate}
\label{tab:answerability_gate}
\begin{tabular}{lrrr}
\toprule
Setting & Answer Rate (\%) & Abstain Rate (\%) & Correctness (\%) \\
\midrule
Without Gate & 86.0000 & 14.0000 & 72.0000 \\
With Gate & 80.0000 & 20.0000 & 80.0000 \\
\bottomrule
\end{tabular}
\end{table}



## 10) Save outputs

Cell này lưu kết quả ra CSV theo đúng yêu cầu.

In [42]:
# 10) Save outputs
retrieval_csv_path = outputs_dir / "retrieval_results.csv"
gate_csv_path = outputs_dir / "gate_results.csv"

retrieval_results_df.to_csv(retrieval_csv_path, index=False)
gate_results_df.to_csv(gate_csv_path, index=False)

print("Saved output files:")
print(f"- {retrieval_csv_path}")
print(f"- {gate_csv_path}")
print(f"- {retrieval_tex_path}")
print(f"- {gate_tex_path}")

Saved output files:
- /home/chplay2020/src/vn-yhct-rag/rag-yhct/notebooks/outputs/retrieval_results.csv
- /home/chplay2020/src/vn-yhct-rag/rag-yhct/notebooks/outputs/gate_results.csv
- /home/chplay2020/src/vn-yhct-rag/rag-yhct/notebooks/outputs/retrieval_results.tex
- /home/chplay2020/src/vn-yhct-rag/rag-yhct/notebooks/outputs/gate_results.tex


## 11) Final summary

Bạn đã có đầy đủ pipeline đánh giá cho 2 bảng:

- **Table I**: Hit@10, MRR cho 4 cấu hình retrieval
- **Table II**: Answer Rate, Abstain Rate, Correctness cho 2 cấu hình gate

Để đánh giá dữ liệu thực tế, hãy sửa các file đầu vào:

- `test_questions.csv`
- `retrieval_predictions.json`
- `gate_predictions.csv`

Sau đó chạy lại notebook từ trên xuống để cập nhật kết quả CSV/LaTeX.